In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
!pip install sentence-transformers scikit-sklearn openai pandas tiktoken


ERROR: Could not find a version that satisfies the requirement scikit-sklearn (from versions: none)
ERROR: No matching distribution found for scikit-sklearn


In [ ]:

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from openai import OpenAI
import pickle
from functools import reduce
import json
import tiktoken
import nltk

nltk.download('punkt')

import time
ENG_PROMPT = f"The following are news articles regarding the Russo-Ukranian war. What is the main common topic they all describe? Answer in 5-10 words. Avoid using words such as 'war', 'russia', 'ukraine'."
ENG_PATH = '/content/gdrive/MyDrive/RUWA_new.csv'
saved_topic_assignments =  '/content/gdrive/MyDrive/english_data/bertopic_topic_assignments.csv'
slo_saved_topic_assignments = '/content/gdrive/MyDrive/slovene_data_2025/bertopic_topic_assignments_1511.csv'
SLO_PATH = "/content/gdrive/MyDrive/slo_data/slovene_data.csv"
OPEN_API_KEY = "SECRET"
class LLM:
  def __init__(self, lang, path, num_topics):
    self.lang = lang
    self.path = path
    self.num_topics = num_topics
    self.docs = None
    self.gpt_labels = []
    self.model = OpenAI(api_key=OPEN_API_KEY)
    self.docs_with_topics = None
    self.failed_gpt_labels = []
  @staticmethod
  def get_embedding_model():
    model_name = "all-MiniLM-L6-v2"
    embedding_model = SentenceTransformer(model_name)
    return embedding_model

  def read_data(self):
    if self.lang == "en":
      texts = pd.read_csv(self.path)
      df = pd.DataFrame(texts, columns=["content"])
      self.docs = df
    else:
      fl = open(self.path, 'rb')
      self.docs= pickle.load(fl)

  @staticmethod
  def get_uniq_labels(labels):
    if len(labels)==0:
      return ""

    return ", ".join(set(reduce(list.__add__, map(lambda x: x.split(), labels))))

  def embed_and_cluster(self):
    embedding_model = self.get_embedding_model()
    embeddings = embedding_model.encode(self.docs['content'].tolist(), show_progress_bar=True)

    print("Clustering with KMeans...")
    kmeans = KMeans(n_clusters=self.num_topics, random_state=42)
    self.docs['cluster'] = kmeans.fit_predict(embeddings)


  def chunk_text(self, text, chunk_size=2000, overlap=200):
    enc = tiktoken.encoding_for_model("gpt-4o-mini")
    tokens = enc.encode(text)

    chunks = []
    start = 0

    while start < len(tokens):
        end = start + chunk_size
        chunk = tokens[start:end]
        chunks.append(enc.decode(chunk))
        start += chunk_size - overlap

    return chunks


  def load_docs_with_topics(self, file_path):
      """
      Loads the dataframe with documents, dates, and topics from a file.
      Assumes the file is a CSV with 'Document', 'Date', and 'Topic' columns.
      """
      try:
          self.docs_with_topics = pd.read_csv(file_path, dayfirst=True)
          # Ensure 'Date' column is in datetime format
          if self.lang=="en":
            self.docs_with_topics['Date'] = pd.to_datetime(self.docs_with_topics['Date'], errors='coerce')
          else:
            self.docs_with_topics['Date'] = pd.to_datetime(self.docs_with_topics['Date'],  format='%d-%m-%Y', errors='coerce')

          if 'Topic' not in self.docs_with_topics.columns:
              self.docs_with_topics['Topic'] = [1 for _ in range(0, len(self.docs_with_topics))]

          if 'Document' not in self.docs_with_topics.columns and 'content' in self.docs_with_topics.columns:
              self.docs_with_topics['Document'] = self.docs_with_topics['content']

      except FileNotFoundError:
          print(f"Error: File not found at {file_path}")
          self.docs_with_topics = pd.DataFrame() # Create an empty DataFrame to avoid further errors
      except Exception as e:
          print(f"Error loading data from {file_path}: {e}")
          self.docs_with_topics = pd.DataFrame()

  def get_prompt(self, representative_texts):
      return f"""
      Identify and extract significant events from a provided set of news documents about the Russo-Ukrainian war, ensuring each event is clearly linked to its date as indicated in the documents. The input is a pre-clustered list of news articles, meaning all documents within a single batch pertain to related topics or events.

Work through the following approach:

- Carefully read each article in the cluster and identify notable actions, decisions, incidents, or outcomes relevant to the Russo-Ukrainian war timeline.
- For each identified event, determine the correct date from the document(s) and associate the event with that date.
- For overlapping or repeated coverage from different articles on the same date, synthesize the information, avoiding redundant events, and combine relevant details for clarity.
- For multi-day events, assign the event to the initial reported date unless a different date is specified as more appropriate.
- Repeat the process for each date represented in the cluster, working step-by-step to ensure full coverage of all significant events.

Before producing the final output, internally verify that all relevant details from the article set are represented, and that the summary does not omit major unique events.

Output format:
- Provide the result as a JSON array. Each entry should have:
    - "date": The event date in YYYY-MM-DD format (from the documents).
    - "event_summary": A brief but complete description of the significant event(s) that occurred on that date (up to 10 words), written in clear and neutral terms.

Example Format:
[
  {{
    "date": "2022-05-16",
    "event_summary": "Avoz steel evacuation."
  }},
  {{
    "date": "2022-05-18",
    "event_summary": "Finland and Sweden's application for joining NATO."
  }}
]
(Real responses should be longer and reflect the number of significant dates/events in the cluster. Use placeholder content in this example.)

Important reminders:
- Summarize, do not copy, article text.
- Every extracted event must have a clearly assigned date.
- Exclude minor developments or redundant details unless they reflect major shifts or outcomes.
- Use neutral, factual language without speculation or emotive phrasing.
- The event summary should be short and no more than 10 words.
- If the date doesn't have some specific event or something not of interest, no need to include it.
- One event per date
- The results are output in a valid JSON format
- {"Results should be in English " if self.lang =="en" else "Results should be in Slovene" }

[REMINDER: Your objective is to extract and present significant events (with dates) from related clusters of Russo-Ukrainian war news articles. Output as detailed above. {"Results should be in English " if self.lang =="en" else "Results should be in Slovene" }]"""



  def build_concat_chunks(self, df, date_col="Date", content_col="Document",
                        max_tokens=2000, overlap=200):

    enc = tiktoken.encoding_for_model("gpt-4o-mini")

    def chunk_text(text, chunk_size=max_tokens, overlap=overlap):
        tokens = enc.encode(text)
        chunks = []
        start = 0
        while start < len(tokens):
            end = start + chunk_size
            chunk = tokens[start:end]
            chunks.append(enc.decode(chunk))
            start += chunk_size - overlap
        return chunks

    final_chunks = []
    current_text = ""
    current_tokens = 0
    for _, row in df.iterrows():
        block = f"BEGIN \n\n Date: {row[date_col]}\nContent: {row[content_col]}\n\n \n END \n\n"
        block_tokens = len(enc.encode(block))

        # If adding this block exceeds the token limit → finalize current chunk
        if current_tokens + block_tokens > max_tokens:
            # If the chunk itself is still too large, chunk it
            if current_tokens > max_tokens:
                final_chunks.extend(chunk_text(current_text))
            else:
                final_chunks.append(current_text)

            # Start new chunk
            current_text = block
            current_tokens = block_tokens
        else:
            # Safe to append to current chunk
            current_text += block
            current_tokens += block_tokens

    # Append the last chunk
    if current_text:
        if current_tokens > max_tokens:
            final_chunks.extend(chunk_text(current_text))
        else:
            final_chunks.append(current_text)

    return final_chunks

  def get_gpt_events_by_dates(self, ranges):

    for start_date, end_date in ranges:

            topic_df = self.docs_with_topics.copy()
            # Filter by date range if specified
            if start_date and end_date:
                try:
                    start_dt = pd.to_datetime(start_date)
                    end_dt = pd.to_datetime(end_date)
                    topic_df = topic_df[(topic_df['Date'] >= start_dt) & (topic_df['Date'] <= end_dt)]
                    topic_df = self.get_content_by_topic_and_date(topic_df)
                    print(f'checking {start_date}, {end_date}, len: {len(topic_df)}')
                except ValueError:
                    print("Invalid date format. Please use YYYY-MM-DD.")
                    continue


            chunks = self.build_concat_chunks(df=topic_df, max_tokens=150000)

            for chunk in chunks:
              prompt = self.get_prompt(chunk) # Pass existing labels to avoid repetition

              try:
                  response = self.model.responses.create(
                      model="gpt-4.1-nano",
                      input=[{
                        "role": "system",
                        "content": [
                          {
                            "type": "input_text",
                            "text": prompt
                          }
                        ]
                      },
                            {
                                "role": "user",
                                "content": [
                                {
                                  "type": "input_text",
                                  "text": chunk
                                }
                              ]
                            }],
                      temperature=0.3
                  )
                  label = response.output_text
              except Exception as e:
                  print(f"Error with topic : {e}")
                  label = f"Unknown (Error: {e})"

              try:
                self.gpt_labels.append(json.loads(label))
              except:
                self.failed_gpt_labels.append(label)

              time.sleep(10)

  def get_gpt_topics(self, ranges):
      """
      Generates GPT topics based on documents within a specified date range for each topic.

      Args:
          start_date (str, optional): The start date for filtering documents (YYYY-MM-DD). Defaults to None.
          end_date (str, optional): The end date for filtering documents (YYYY-MM-DD). Defaults to None.
      """
      if self.docs_with_topics is None or self.docs_with_topics.empty:
          print("docs_with_topics DataFrame is not loaded or is empty.")
          return

      unique_topics = self.docs_with_topics['Topic'].unique()
      self.gpt_labels = {str(t): [] for t in unique_topics}
      self.failed_gpt_labels = {str(t): [] for t in unique_topics}
      for topic_id in unique_topics:
          # Filter by topic

          for start_date, end_date in ranges:

            topic_df = self.docs_with_topics[self.docs_with_topics['Topic']==topic_id].copy()
            # Filter by date range if specified
            if start_date and end_date:
                try:
                    start_dt = pd.to_datetime(start_date)
                    end_dt = pd.to_datetime(end_date)
                    topic_df = topic_df[(topic_df['Date'] >= start_dt) & (topic_df['Date'] <= end_dt)]
                    topic_df = self.get_content_by_topic_and_date(topic_df)
                    print(f'checking {start_date}, {end_date}, len: {len(topic_df)}')
                except ValueError:
                    print("Invalid date format. Please use YYYY-MM-DD.")
                    continue


            if topic_df.empty:
                print(f"No documents found for topic {topic_id} in the specified date range.")
                #self.gpt_labels[str(topic_id)].append(f"Unknown (No data for topic {topic_id} in date range)")
                continue

            # Concatenate texts with dates
            representative_texts = ""
            for index, row in topic_df.iterrows():
              representative_texts += f" Date: {row['Date'].strftime('%Y-%m-%d')} \n Document: {row['Document']} \n ------ "



            prompt = self.get_prompt(representative_texts) # Pass existing labels to avoid repetition

            try:
                response = self.model.responses.create(
                    model="gpt-4.1-nano",
                    input=[{
                      "role": "system",
                      "content": [
                        {
                          "type": "input_text",
                          "text": prompt
                        }
                      ]
                    },
                          {
                              "role": "user",
                              "content": [
                              {
                                "type": "input_text",
                                "text": representative_texts
                              }
                            ]
                          }],
                    temperature=0.3
                )
                label = response.output_text
            except Exception as e:
                label = f"Unknown (Error: {e})"

            try:
              self.gpt_labels[str(topic_id)].append(json.loads(label))
            except:
              self.failed_gpt_labels[str(topic_id)].append(label)

            time.sleep(10)

  def save_topic_assignments(self, identifier=""):
    import pickle
    with open(f'/content/gdrive/MyDrive/pickles/gpt_topic_assignments{identifier}.pkl', 'wb') as to_save:
      pickle.dump(self.docs[['cluster','topic_label']], to_save)

  def get_content_by_topic_and_date(self, df):
      """
      Filters the class's DataFrame by topic assignment number, groups by date, and returns the first content for each date.

      Args:
          topic_assignment_number (int): The numerical topic assignment to filter by.

      Returns:
          pd.DataFrame: A DataFrame with 'date' and 'content' columns, containing the first content for each date within the specified topic.
      """


      # 3. Group and Extract
      grouped_content = df.groupby('Date')['Document'].sum().reset_index()
      #grouped_content = df.groupby('Date')['Document'].transform(lambda x: ','.join(x))
      return grouped_content


  def run(self):
    self.embed_and_cluster()
    # The original call to get_gpt_topics used self.docs, which is not what
    # the user wants now. The user wants to use self.docs_with_topics.
    # This run method might need to be adjusted based on the overall workflow.
    # For now, I will not call get_gpt_topics here to avoid errors, as
    # self.docs_with_topics is not loaded by read_data.
    # self.get_gpt_topics(self.docs)
    pass

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
l = LLM('slo', slo_saved_topic_assignments, 6)

In [ ]:
l.load_docs_with_topics(slo_saved_topic_assignments)

In [ ]:
slo_dates = [('2022-01-01', '2022-02-28'),('2022-03-01', '2022-04-30'), ('2022-05-01', '2022-06-30'), ('2022-07-01', '2022-09-30'), ('2022-09-30', '2022-11-30'), ('2022-12-01', '2023-02-28'),('2023-03-01', '2023-04-30'), ('2023-05-01', '2023-06-30'), ('2023-07-01', '2023-09-30'), ('2023-09-30', '2023-11-30'), ('2023-12-01', '2024-02-28'), ('2024-03-01', '2024-06-30'), ('2024-07-01', '2024-09-30'),  ('2024-10-01', '2024-12-31')]
eng_dates = [('2022-01-01', '2022-02-28'),('2022-03-01', '2022-04-30'), ('2022-05-01', '2022-06-30'), ('2022-07-01', '2022-09-30'), ('2022-09-30', '2022-11-30'), ('2022-12-01', '2023-02-28'),('2023-03-01', '2023-04-30'), ('2023-05-01', '2023-06-30'), ('2023-07-01', '2023-09-30'), ('2023-09-30', '2023-11-30'),  ]
l.get_gpt_topics(slo_dates)


checking 2022-01-01, 2022-02-28, len: 9
checking 2022-03-01, 2022-04-30, len: 49
checking 2022-05-01, 2022-06-30, len: 40
checking 2022-07-01, 2022-09-30, len: 46
checking 2022-09-30, 2022-11-30, len: 42
checking 2022-12-01, 2023-02-28, len: 28
checking 2023-03-01, 2023-04-30, len: 24
checking 2023-05-01, 2023-06-30, len: 19
checking 2023-07-01, 2023-09-30, len: 30
checking 2023-09-30, 2023-11-30, len: 25
checking 2023-12-01, 2024-02-28, len: 20
checking 2024-03-01, 2024-06-30, len: 53
checking 2024-07-01, 2024-09-30, len: 32
checking 2024-10-01, 2024-12-31, len: 19
checking 2022-01-01, 2022-02-28, len: 25
checking 2022-03-01, 2022-04-30, len: 59
checking 2022-05-01, 2022-06-30, len: 59
checking 2022-07-01, 2022-09-30, len: 84
checking 2022-09-30, 2022-11-30, len: 60
checking 2022-12-01, 2023-02-28, len: 56
checking 2023-03-01, 2023-04-30, len: 54
checking 2023-05-01, 2023-06-30, len: 54
checking 2023-07-01, 2023-09-30, len: 83
checking 2023-09-30, 2023-11-30, len: 53
checking 2023-12-

In [ ]:
df = pd.read_csv(slo_saved_topic_assignments)
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y', errors="coerce")


In [ ]:
start_date = pd.to_datetime('2022-01-01')
end_date = pd.to_datetime('2022-02-28')

filtered = df[(df['Date'] >start_date)  & (df['Date'] < end_date)]
l.get_content_by_topic_and_date(filtered)

,Date,Document
0,2022-02-01,"""Predstavljajte si, da je Ukrajina članica Nat..."
1,2022-02-02,Filmski kritiki so Monico Vitti častili predvs...
2,2022-02-03,"""Turčija je pripravljena narediti svoj del za ..."
3,2022-02-04,"""Imamo informacije, da bodo Rusi verjetno ustv..."
4,2022-02-05,Čeprav so moje knjige videti tipično slovenske...
5,2022-02-06,V petek bi prav lahko kakšno razprodajo povzro...
6,2022-02-07,Putin je medtem pohvalil prizadevanja francosk...
7,2022-02-08,"""Naredili mu bomo konec,"" je bil jasen Biden n..."
8,2022-02-09,Razprava bi zdaj lahko potekala več mesecev.Do...
9,2022-02-10,Kot je sporočila litovska premierka Ingrida Ši...


In [ ]:
df[df['Date']=="2022-02-23"]

,Document,Date,Article_id,Topic


In [ ]:
import pickle
with open('/content/gdrive/MyDrive/slovene_data_2025/extracted_event_dump_2354.pkl', 'wb') as f:
   pickle.dump(l.gpt_labels, f)

In [ ]:
l.gpt_labels

{'1': [[{'date': '2022-02-05',
    'event_summary': 'Razgovor z Drago Jančarjem o povezavah s Srbijo.'},
   {'date': '2022-02-07',
    'event_summary': 'Sodišče v Amsterdamu odloči, da se skitsko zlato vrne Ukrajini.'},
   {'date': '2022-02-09',
    'event_summary': 'Kajetan Gantar objavi avtobiografijo Penelopin prt.'},
   {'date': '2022-02-13',
    'event_summary': 'Claire Keegan predstavi novo knjigo Vse te malenkosti.'},
   {'date': '2022-02-15',
    'event_summary': 'Razstava pustnih razglednic in motivov kurenta na Ptuju.'},
   {'date': '2022-02-16',
    'event_summary': 'Obnova in razstava kurentov v Cerknem, ohranjanje tradicije.'},
   {'date': '2022-02-19',
    'event_summary': 'Film o delavskem razredu v Franciji, portret žensk delavskega razreda.'},
   {'date': '2022-02-22',
    'event_summary': 'Organizacija tradicionalne laufarije v Cerknem v živo.'},
   {'date': '2022-02-23',
    'event_summary': 'Razstava pustnih razglednic in izdaja znamk v Ptuju.'}],
  [{'datum': '2022

In [ ]:
new_gpt_labels = { }
for topic_key, list_of_lists_for_key in l.gpt_labels.items():
        new_gpt_labels[topic_key] =  [
        x
        for xs in list_of_lists_for_key
        for x in xs
    ]




In [ ]:
new_df = pd.DataFrame(
    [(k, v['datum'], v['date'], v['event_summary'], v['dogodek']) for k, v in new_gpt_labels.items() ],
    columns=['topic_id', 'datum', 'date', 'event_summary', 'dogodek']
)

TypeError: list indices must be integers or slices, not str

In [ ]:
rows = []

for topic_id, events in l.gpt_labels.items():
    for dates in events:
        for date in dates:
          rows.append({
              'topic_id': int(topic_id),
              'datum': date.get('datum', None),
              'date': date.get('date', None),
              'event_summary': date.get('event_summary', None),
              'dogodek': date.get('dogodek', None)

          })

df = pd.DataFrame(rows)

In [ ]:
df

,topic_id,datum,date,event_summary,dogodek
0,1,None,2022-02-05,Razgovor z Drago Jančarjem o povezavah s Srbijo.,None
1,1,None,2022-02-07,"Sodišče v Amsterdamu odloči, da se skitsko zla...",None
2,1,None,2022-02-09,Kajetan Gantar objavi avtobiografijo Penelopin...,None
3,1,None,2022-02-13,Claire Keegan predstavi novo knjigo Vse te mal...,None
4,1,None,2022-02-15,Razstava pustnih razglednic in motivov kurenta...,None
...,...,...,...,...,...
2120,4,2024-11-20,None,"Izrael napade v Gazi, ubitih več kot 44.000 lj...",None
2121,4,2024-11-22,None,"Hezbolah izstrelil rakete na Izrael, izredno s...",None
2122,4,2024-11-23,None,"Slovenija odloči za nakup izraelskega orožja, ...",None
2123,4,2024-11-24,None,"Izrael napade Gazo, ubitih več kot 44.000 ljudi.",None


In [ ]:
df['date'] =  df["date"].fillna(df["datum"])
df['event_summary'] = df['event_summary'].fillna(df['dogodek'])


In [ ]:
df["date"] = df["date"].replace("", pd.NA).fillna(df["datum"].replace("", pd.NA))
df["event_summary"] = df['event_summary'].replace("", pd.NA).fillna(df["dogodek"].replace("", pd.NA))

In [ ]:
df["date"] = df.get("date", pd.NA).fillna(df.get("datum"), inplace=True)
df["event_summary"] = df.get("event_summary", pd.NA).fillna(df.get("dogodek"), inplace=True)

# Remove old ones
df = df.drop(columns=["datum", "dogodek_summary"], errors="ignore")

In [ ]:
df = df.drop(columns=["datum", "dogodek"])

In [ ]:
df.to_csv('/content/gdrive/MyDrive/slovene_data_2025/extracted_events_by_topics.csv')

In [ ]:
df = df.drop(columns=["datum", "dogodek"])

In [ ]:
df[df['date']=="2023-04-08"]

,topic_id,datum,date,event_summary,dogodek
404,-1,2023-04-08,2023-04-08,"Ruska vojska napade s brezpilotniki, prizadene...","Ruska vojska napade s brezpilotniki, prizadene..."
892,0,2023-04-08,2023-04-08,Ukrajinska vojska napredovala na vzhodu in jug...,Ukrajinska vojska napredovala na vzhodu in jug...


In [ ]:
df.head()

,topic_id,date,event_summary
0,1,,
1,1,,
2,1,,
3,1,,
4,1,,


In [ ]:
import pandas as pd

processed_gpt_labels = {}

for topic_key, list_of_lists_for_key in l.gpt_labels.items():
    # Flatten the list of lists for the current topic key
    flattened_for_key = [item for sublist in list_of_lists_for_key for item in sublist]

    if not flattened_for_key:
        processed_gpt_labels[topic_key] = []
        continue

    temp_df = pd.DataFrame(flattened_for_key)

    # Merge 'date' and 'datum' into a single 'date' column
    if 'datum' in temp_df.columns:
        temp_df['date'] = temp_df['date'].fillna(temp_df['datum'])
        temp_df = temp_df.drop(columns=['datum'], errors='ignore')

    # Merge 'event_summary' and 'dogodek' into a single 'event_summary' column
    if 'dogodek' in temp_df.columns:
        temp_df['event_summary'] = temp_df['event_summary'].fillna(temp_df['dogodek'])
        temp_df = temp_df.drop(columns=['dogodek'], errors='ignore')

    # Convert the 'date' column to datetime objects, coercing errors to NaT
    temp_df['date'] = pd.to_datetime(temp_df['date'], errors='coerce')

    # Drop rows where 'date' or 'event_summary' is null after conversion/merging
    subset_cols = ['date']
    if 'event_summary' in temp_df.columns:
        subset_cols.append('event_summary')
    temp_df.dropna(subset=subset_cols, inplace=True)

    # Convert the processed DataFrame back to a list of dictionaries
    processed_gpt_labels[topic_key] = temp_df.to_dict('records')

display(processed_gpt_labels)

In [ ]:
flattened_gpt_labels = flat_list = [
    events
    for events_coll in l.gpt_labels
    for events in events_coll
]

In [ ]:
flattened_gpt_labels

[{'datum': '2022-02-01',
  'dogodek': 'Putin prvič komentiral napetosti med Rusijo in Zahodom.'},
 {'datum': '2022-02-02',
  'dogodek': 'Rusija začela vojaške vaje z Belorusijo, sodelovalo 30.000 vojakov.'},
 {'datum': '2022-02-03',
  'dogodek': 'Putin in Scholz razpravljala o deeskalaciji na meji z Ukrajino.'},
 {'datum': '2022-02-04',
  'dogodek': 'Biden zagotovil, da se bo Rusija soočila s sankcijami, če napade Ukrajino.'},
 {'datum': '2022-02-05',
  'dogodek': 'ZDA začasno umaknile večino osebja iz veleposlaništva v Kijevu.'},
 {'datum': '2022-02-06',
  'dogodek': 'Nato in ZDA opozorile na povečano kopičenje ruskih sil ob meji z Ukrajino.'},
 {'datum': '2022-02-07',
  'dogodek': 'Putin in Macron razpravljala o možnostih deeskalacije krize.'},
 {'datum': '2022-02-08',
  'dogodek': 'Scholz in Scholz v Moskvi pozvala Rusijo k umiku enot z meje.'},
 {'datum': '2022-02-09',
  'dogodek': 'Biden in Zelenski ponovno potrdila podporo Ukrajini, opozorila na ruske provokacije.'},
 {'datum': '

In [ ]:
df = pd.DataFrame(flattened_gpt_labels)


In [ ]:
df["date"] = df.get("date", pd.NA).fillna(df.get("datum"))
df["event_summary"] = df.get("event_summary", pd.NA).fillna(df.get("dogodek"))

# Remove old ones
df = df.drop(columns=["datum", "dogodek"])

In [ ]:
df

,event_summary,date
0,Putin prvič komentiral napetosti med Rusijo in...,2022-02-01
1,"Rusija začela vojaške vaje z Belorusijo, sodel...",2022-02-02
2,Putin in Scholz razpravljala o deeskalaciji na...,2022-02-03
3,"Biden zagotovil, da se bo Rusija soočila s san...",2022-02-04
4,ZDA začasno umaknile večino osebja iz veleposl...,2022-02-05
...,...,...
1186,Ruska in sirska vojska izvajata zračne napade ...,2024-11-29
1187,Civilne žrtve in razseljevanje zaradi spopadov...,2024-11-29
1188,Razprava o pomenu jezika in identitete v migra...,2024-11-30
1189,Pisatelj Ferić o pomenu jezika in spominu v so...,2024-11-30


In [ ]:
df.to_csv('/content/gdrive/MyDrive/slovene_data_2025/extracted_events_final_maybe.csv')

In [ ]:
# This cell is no longer needed as its logic has been integrated into 1jprBZ6PdzON.

In [ ]:
# This cell is no longer needed as its logic has been integrated into 1jprBZ6PdzON and the output is now a dictionary, not a single flattened DataFrame.

In [ ]:
# This cell is no longer needed as its logic has been integrated into 1jprBZ6PdzON.

In [ ]:
# This cell is no longer needed as its logic has been integrated into 1jprBZ6PdzON and the output is now a dictionary, not a single flattened DataFrame.

In [ ]:
# Convert the flattened list of dictionaries to a pandas DataFrame
#flattened_df = pd.DataFrame(flattened_gpt_labels)

# Specify the output CSV file path
output_csv_path = '/content/gdrive/MyDrive/slovene_data_2025/extracted_event_4_test_broken.csv'
df.to_csv(output_csv_path)
# To save the processed_gpt_labels dictionary, you might want to save it as a JSON file
# import json
# with open(output_csv_path.replace('.csv', '.json'), 'w') as f:
#    json.dump(processed_gpt_labels, f, indent=4, default=str)

print(f"Processing complete. Output is stored in 'processed_gpt_labels' dictionary.")


Processing complete. Output is stored in 'processed_gpt_labels' dictionary.


In [ ]:
l.run()

Batches:   0%|          | 0/243 [00:00<?, ?it/s]

Clustering with KMeans...


In [ ]:
l.gpt_labels

{'1': [[{'date': '2022-01-03',
    'event_summary': 'Maribor praznuje pustno tradicijo in kulturne prireditve.'},
   {'date': '2022-01-04',
    'event_summary': 'Luke Koper beleži povečan pretovor, a težave v logistiki.'},
   {'date': '2022-01-07',
    'event_summary': 'Film Top Gun: Maverick odloženo zaradi pandemije, uspeh pričakovan.'},
   {'date': '2022-01-11',
    'event_summary': 'Film o Freudovi zgodovini bo režiral Matt Brown.'},
   {'date': '2022-02-03',
    'event_summary': 'Ob 60-letnici Lojzeta Krajnčana koncert Big Banda RTV Slovenija.'},
   {'date': '2022-02-04',
    'event_summary': 'Mednarodni festival Schamrock v Münchnu s pesniškim dialogom.'},
   {'date': '2022-02-11',
    'event_summary': 'Režiser Matt Brown pripravlja film o Freudovi osebni zgodovini.'}],
  [{'datum': '2022-03-03',
    'dogodek': 'Predstavitev knjige Vojnovića Zbiralec strahov in razmišljanje o miru.'},
   {'datum': '2022-03-03',
    'dogodek': 'Razprava o aktualnih dogodkih v Ukrajini in Putinovi 

In [ ]:

# 5. CREATE TOPIC LABELS WITH GPT
topic_labels = []

for cluster_id in range(num_topics):
    cluster_texts = df[df['cluster'] == cluster_id]['content'].tolist()
    representative_texts = "\n".join(cluster_texts[:30])  # top 20 examples

    prompt = f"""The following are news articles regarding the Russo-Ukranian war. What is the main common topic they all describe? Answer in 5-10 words. Avoid using words such as 'war', 'russia', 'ukraine'. In addition, the label should not contain words from one of the followings: {', '.join(topic_labels)}

{representative_texts}

Topic:"""

    try:
        response = openai.completions.create(
            model="gpt-4.1-nano",
            prompt=prompt,
            temperature=0.6
        )
        label = response.choices[0].text.strip()
    except Exception as e:
        print(f"Error with cluster {cluster_id}: {e}")
        label = "Unknown"

    topic_labels.append(label)

# 6. ASSIGN TOPIC LABELS TO DOCUMENTS
label_map = {i: label for i, label in enumerate(topic_labels)}
df['topic_label'] = df['cluster'].map(label_map)

# 7. DISPLAY RESULT
print(df[['content', 'cluster', 'topic_label']].sample(10))

Error with cluster 7: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-nano in organization org-440TFdiILgGW67IGVCCLNc3T on tokens per min (TPM): Limit 200000, Used 138305, Requested 63755. Please try again in 618ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
                                                 content  cluster  \
9722    Russian troops are likely to transfer troops ...        2   
14377  \n    Ukrainian President Volodymyr Zelensky h...        0   
15016  \n    A number of explosions on the territory ...       13   
10588  \n    The sanctions against the Russian Federa...        8   
2724   Japan will restart more idled nuclear plants a...       11   
3167   The United States has accused Moscow of a plot...        1   
14774  \n    In Donetsk region, soldiers of the Natio...        0   
3492   Russian gas producer Gazprom has said it has s...        8   
170

In [ ]:
import pickle
with open('/content/gdrive/MyDrive/pickles/gpt_topic_assignments_1809_2.pkl', 'wb') as to_save:
  pickle.dump(df[['cluster','topic_label']], to_save)

In [ ]:
df['topic_label'].unique()

array(['Misinformation, propaganda, and false claims online.',
       'Impact of conflict on civilians, infrastructure, and international response.',
       'Ongoing military conflicts and strategic military operations.',
       'Civilian casualties and destruction in Ukraine conflicts.',
       'Regional political, economic, and social developments.',
       'International responses to regional conflict and geopolitical tensions.',
       'Unknown',
       'International reactions and responses to regional conflict.',
       'International economic and political impacts of Ukraine conflict.',
       'Regional political and military developments.',
       'International responses and support amid regional conflict.',
       'International responses and support for Ukraine.',
       'Civilian suffering and displacement amid military conflict.',
       'Regional military developments and territorial shifts.'],
      dtype=object)